# Lambertian Shading: Photometric Stereo, the Bas-Relief Ambiguity and Harmonic Images

Prepared for Fall 2026. Reference: the notes for Lecture 17 (Lambertian Models of Reflectance and Rendering). If you find a bug in this notebook, please email the TAs or tell us at office hours.

**Before you start.** Everything is synthetic — a $64 \times 64$ depth map rendered with numpy under fixed seeds, so you get our numbers. Every figure and number a question asks for must appear in your PDF (eleven figures and about twenty numbers in all): to vary a parameter, copy the cell below the question (or loop) rather than re-running it above.

---

**The Lambertian model.** A pixel $\vec x$ with albedo $a(\vec x)$ and unit normal $\vec n(\vec x)$, lit by a distant light $\vec s$, has the intensity $I(\vec x) = a(\vec x)\,\vec n(\vec x) \cdot \vec s$, linear in the light as long as no point faces away from it. A point with $a\,\vec n \cdot \vec s < 0$ is in attached shadow and has intensity 0, so the model with each light's contribution clipped at zero,

$$I(\vec x) = \sum_\mu \max\{a(\vec x)\,\vec n(\vec x) \cdot \vec s_\mu,\, 0\},$$

is no longer linear; cast shadows are not in this formula. The scaled normal is $\vec b = a\,\vec n$ ($a = |\vec b|$, $\vec n = \vec b / |\vec b|$), and a depth map $z(x, y)$ has $\vec n = (-z_x, -z_y, 1) / \sqrt{z_x^2 + z_y^2 + 1}$.

## The synthetic surface and the renderer

An elliptical dome on a flat background, $z = 0.9 \sqrt{\max\{0,\, 1 - (x/0.85)^2 - (y/0.95)^2\}}$ on a $64 \times 64$ grid of $x, y \in [-1, 1]$, plus two Gaussian bumps (height 0.08, width 0.15); albedo 0.9 with a darker patch (0.5) and a stripe (0.6). `mask` is the dome's footprint: background pixels are left out of every matrix and every error. `render` gives one image per light — the clipped sum, or the linear model with `clip=False`.

In [ ]:
from __future__ import annotations
import heapq

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre
from scipy import ndimage, sparse
from scipy.sparse.linalg import lsqr

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)

# The colours of the lecture notes
BLUE, ORANGE, TEAL = "#1f77b4", "#e6550d", "#1b9e77"     # the linear model; the clipped sum (with shadows); the kernel coefficients
RESIDUAL_CMAP = "RdBu_r"                                  # red positive, blue negative, white zero
N_GRID = 64
KERNEL_ENERGY = 2 * np.pi / 3                             # the energy of the Lambertian kernel, int_{S^2} k^2 = sum_n k_n^2
OPTIONAL_CAST_SHADOWS = False                             # True adds cast shadows to the renderings of the SVD section (slower)
fmt4 = {"float_kind": lambda v: f"{v:.4g}"}               # four significant digits in printed arrays


def rms(a, b=0.0, where=None) -> float:
    """Root mean square of a - b over the pixels where `where` is True (all pixels if None)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if where is not None:
        d = d[np.asarray(where, dtype=bool)]
    return float(np.sqrt(np.mean(d ** 2))) if d.size else float("nan")


def spacing(n: int) -> float:
    """The grid spacing h of n samples of [-1, 1]."""
    return 2.0 / (n - 1)


# ------------------------------------------------------------------ 1. The surface, its normals and the renderer

def make_surface(n: int = N_GRID, bumps: bool = True):
    """The synthetic surface on x, y in [-1, 1] (n x n pixels; axis 0 is y, axis 1 is x, both increasing):
    an elliptical dome z = 0.9 sqrt(max(0, 1 - (x/0.85)^2 - (y/0.95)^2)) with, if bumps, two Gaussian bumps of height 0.08
    and width 0.15 at (0.3, 0.2) and (-0.35, -0.3); albedo 0.9 on the dome with a 12 x 10-pixel patch of 0.5 and a stripe of
    0.6, 0 on the background. Returns x, y, z, albedo and the boolean mask (the dome's footprint)."""
    h = spacing(n)
    x, y = np.meshgrid(np.linspace(-1.0, 1.0, n), np.linspace(-1.0, 1.0, n))
    inside = 1.0 - (x / 0.85) ** 2 - (y / 0.95) ** 2
    mask = inside > 0
    z = 0.9 * np.sqrt(np.maximum(inside, 0.0))
    if bumps:
        for cx, cy in ((0.3, 0.2), (-0.35, -0.3)):
            z += 0.08 * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * 0.15 ** 2)) * mask
    albedo = np.where(mask, 0.9, 0.0)
    albedo[mask & (np.abs(x + 0.25) < 6 * h) & (np.abs(y - 0.35) < 5 * h)] = 0.5        # the 12 x 10 patch
    albedo[mask & (np.abs(x - y + 0.45) < 0.05)] = 0.6                                 # the stripe
    return x, y, z, albedo, mask


def normals_from_depth(z, h: float | None = None):
    """Unit normals n = (-z_x, -z_y, 1) / sqrt(z_x^2 + z_y^2 + 1) of a depth map, with the slopes by numpy.gradient
    (central differences, spacing h). Returns an (n, n, 3) array; n_3 > 0 everywhere."""
    z = np.asarray(z, dtype=float)
    h = spacing(z.shape[0]) if h is None else h
    z_y, z_x = np.gradient(z, h)                            # axis 0 is y, axis 1 is x
    n = np.stack([-z_x, -z_y, np.ones_like(z)], axis=-1)
    return n / np.linalg.norm(n, axis=-1, keepdims=True)


def unit(b):
    """The unit normals b / |b| of scaled normals, with (0, 0, 1) where b = 0 (the background)."""
    norm = np.linalg.norm(b, axis=-1, keepdims=True)
    out = np.divide(b, norm, out=np.zeros_like(b, dtype=float), where=norm > 0)
    out[..., 2] = np.where(norm[..., 0] > 0, out[..., 2], 1.0)
    return out


def render(b, lights, clip: bool = True):
    """Images of the scaled normals b (n, n, 3) under one light (3,) or several lights (N, 3), one image per light:
    max{b . s, 0} (the clipped sum, one term per light; the image under several lights at once is the sum of the images)
    or, with clip=False, the linear model b . s. Returns (n, n) for one light and (N, n, n) for N lights."""
    lights = np.asarray(lights, dtype=float)
    images = np.einsum("yxi,mi->myx", np.asarray(b, dtype=float), np.atleast_2d(lights))
    if clip:
        images = np.maximum(images, 0.0)
    return images[0] if lights.ndim == 1 else images


def render_cast_shadows(z, b, lights, x, y, n_steps: int = 80):
    """The clipped sum with cast shadows added: a pixel is dark under a light when the ray from its surface point toward the
    light passes below the surface (checked by stepping along the ray and interpolating the depth). Optional and slower."""
    one = np.ndim(lights) == 1
    lights = np.atleast_2d(np.asarray(lights, dtype=float))
    h = x[0, 1] - x[0, 0]
    images = render(b, lights)
    step = 1.5 * h
    for m, s in enumerate(lights):
        blocked = np.zeros(z.shape, dtype=bool)
        for k in range(1, n_steps + 1):
            t = k * step
            px, py, pz = x + t * s[0], y + t * s[1], z + t * s[2]
            rows, cols = (py + 1.0) / h, (px + 1.0) / h
            on_grid = (rows >= 0) & (rows <= z.shape[0] - 1) & (cols >= 0) & (cols <= z.shape[1] - 1)
            z_surface = ndimage.map_coordinates(z, [np.clip(rows, 0, z.shape[0] - 1), np.clip(cols, 0, z.shape[1] - 1)], order=1)
            blocked |= on_grid & (z_surface > pz + 1e-9)
        images[m][blocked] = 0.0
    return images[0] if one else images


def random_lights(n: int, min_s3: float = 0.0, seed: int = 0):
    """n random unit light directions on the hemisphere s_3 > 0 (Gaussian vectors normalised, s_3 made positive), keeping only
    those with s_3 >= min_s3 (min_s3 = cos 60 degrees keeps the lights within 60 degrees of the viewing direction; min_s3 is a
    cosine and must lie in [0, 1)). Returns (n, 3)."""
    if not 0.0 <= min_s3 < 1.0:
        raise ValueError(f"min_s3 is the cosine of the largest angle to the viewing direction and must lie in [0, 1); got {min_s3}")
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        s = rng.normal(size=(2 * n, 3))
        s /= np.linalg.norm(s, axis=1, keepdims=True)
        s[:, 2] = np.abs(s[:, 2])
        out.extend(s[s[:, 2] >= min_s3])
    return np.array(out[:n])


# ------------------------------------------------------------------ 2. The image matrix and its singular values

def image_matrix(images, mask):
    """J (pixels x images): the N images (N, n, n) restricted to the mask pixels, one image per column."""
    return np.asarray(images)[:, mask].T


def svd_curve(J):
    """The singular values D_11 >= D_22 >= ... of J, their ratios D_ii / D_11 and the cumulative energy
    sum_{i <= k} D_ii^2 / sum_i D_ii^2 for k = 1, 2, ... Returns (D, ratios, cumulative)."""
    d = np.linalg.svd(np.asarray(J, dtype=float), compute_uv=False)
    return d, d / d[0], np.cumsum(d ** 2) / np.sum(d ** 2)


def components_for(cumulative, fraction: float) -> int:
    """The smallest k whose cumulative energy reaches the fraction."""
    return int(np.argmax(np.asarray(cumulative) >= fraction)) + 1


# ------------------------------------------------------------------ 3. Photometric stereo, depth from normals, the GBR

def photometric_stereo(images, lights, mask):
    """Least squares for b at every mask pixel from its intensities under the lights that do not put it in attached shadow
    (I > 0; a shadowed intensity does not obey I = b . s). Pixels with fewer than three such lights are marked invalid
    (b = 0 there). Returns the albedo |b| (n, n), the unit normals b / |b| (n, n, 3), the boolean valid (n, n) and b (n, n, 3)."""
    images, lights = np.asarray(images, dtype=float), np.asarray(lights, dtype=float)
    lit = (images > 0) & mask[None]
    W = lit.astype(float)
    LtWL = np.einsum("myx,mi,mj->yxij", W, lights, lights)             # per pixel: sum over its lit lights of s s^T
    LtWm = np.einsum("myx,myx,mi->yxi", W, images, lights)             # per pixel: sum of I s
    valid = mask & (lit.sum(axis=0) >= 3) & (np.abs(np.linalg.det(LtWL)) > 1e-12)
    if not valid.any():
        raise ValueError("no pixel is lit by three or more lights that are not coplanar: photometric stereo needs f >= 3 such lights "
                         f"({len(lights)} given)")
    b = np.zeros(images.shape[1:] + (3,))
    b[valid] = np.linalg.solve(LtWL[valid], LtWm[valid][..., None])[..., 0]
    albedo = np.linalg.norm(b, axis=-1)
    normals = np.divide(b, albedo[..., None], out=np.zeros_like(b), where=albedo[..., None] > 0)
    return albedo, normals, valid, b


def angular_error(n_est, n_true, valid):
    """The angle in degrees between the estimated and the true unit normals at the valid pixels (0 elsewhere)."""
    cos = np.clip(np.sum(np.asarray(n_est) * np.asarray(n_true), axis=-1), -1.0, 1.0)
    return np.where(valid, np.degrees(np.arccos(cos)), 0.0)


def depth_from_slopes(p, q, mask, h: float | None = None):
    """Depth z by least squares on the slope equations z(x+1, y) - z(x, y) = h p and z(x, y+1) - z(x, y) = h q over every pair
    of neighbouring mask pixels (the slope of a pair is the mean of its two pixels' slopes), solved with scipy's sparse
    least squares (lsqr). The depth is fixed only up to a constant: the returned z has mean 0 over the mask (NaN outside)."""
    mask = np.asarray(mask, dtype=bool)
    n_rows, n_cols = mask.shape
    h = spacing(n_rows) if h is None else h
    index = -np.ones(mask.shape, dtype=int)
    index[mask] = np.arange(mask.sum())
    rows, cols, vals, rhs, k = [], [], [], [], 0
    for dy, dx, slope in ((0, 1, np.asarray(p, dtype=float)), (1, 0, np.asarray(q, dtype=float))):
        pair = mask[:n_rows - dy, :n_cols - dx] & mask[dy:, dx:]
        i0, i1 = index[:n_rows - dy, :n_cols - dx][pair], index[dy:, dx:][pair]
        s = 0.5 * (slope[:n_rows - dy, :n_cols - dx][pair] + slope[dy:, dx:][pair])
        m = len(i0)
        rows += [np.arange(k, k + m), np.arange(k, k + m)]
        cols += [i1, i0]
        vals += [np.ones(m), -np.ones(m)]
        rhs.append(h * s)
        k += m
    A = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(k, int(mask.sum())))
    solution = lsqr(A, np.concatenate(rhs), atol=1e-10, btol=1e-10, iter_lim=20000)[0]
    z = np.full(mask.shape, np.nan)
    z[mask] = solution - solution.mean()
    return z


def integrability_residual(normals, region, h: float | None = None):
    """The residual d/dx (n_2/n_3) - d/dy (n_1/n_3) by central differences (numpy.gradient); the ratios are set to 0 where
    |n_3| < 1e-9. Returned with NaN outside `region` (pixels whose neighbours are all in the mask)."""
    normals = np.asarray(normals, dtype=float)
    h = spacing(normals.shape[0]) if h is None else h
    n3 = normals[..., 2]
    safe = np.abs(n3) > 1e-9
    r1 = np.divide(normals[..., 0], n3, out=np.zeros_like(n3), where=safe)
    r2 = np.divide(normals[..., 1], n3, out=np.zeros_like(n3), where=safe)
    residual = np.gradient(r2, h, axis=1) - np.gradient(r1, h, axis=0)
    return np.where(region, residual, np.nan)


def gbr_matrices(lam_p: float, mu: float, nu: float):
    """The matrices of the GBR with tau = 1 (so lambda = lambda', alpha = -mu, beta = -nu): M, acting on b, with rows
    (lambda', 0, -mu), (0, lambda', -nu), (0, 0, 1), and G, with rows (1, 0, 0), (0, 1, 0), (mu, nu, lambda'), so that the
    lights move to M^{-T} s = G s / lambda."""
    M = np.array([[lam_p, 0.0, -mu], [0.0, lam_p, -nu], [0.0, 0.0, 1.0]])
    G = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [mu, nu, lam_p]])
    return M, G


def gbr(z, b, lights, x, y, lam_p: float, mu: float, nu: float):
    """The generalised bas-relief transformation with tau = 1: the depth z' = lambda' z + mu x + nu y, the scaled normals
    b' = M b (albedo |M b|, normals M b / |M b|) and the lights s' = G s / lambda. Returns z', b' (n, n, 3), lights' (N, 3)."""
    M, G = gbr_matrices(lam_p, mu, nu)
    z_new = lam_p * np.asarray(z, dtype=float) + mu * x + nu * y
    b_new = np.asarray(b, dtype=float) @ M.T
    lights_new = (np.atleast_2d(np.asarray(lights, dtype=float)) @ G.T) / lam_p
    return z_new, b_new, lights_new


# ------------------------------------------------------------------ 4. The Lambertian kernel and the harmonic images

def kernel_coefficients(n_max: int = 8, n_nodes: int = 64):
    """k_n = 2 pi int_0^1 t P_n(t) sqrt((2n+1) / 4 pi) dt for n = 0, ..., n_max (t = cos theta), by Gauss-Legendre numerical
    integration on [0, 1] with n_nodes nodes (exact for these polynomials). Returns the array of k_n."""
    nodes, weights = legendre.leggauss(n_nodes)
    t, w = 0.5 * (nodes + 1.0), 0.5 * weights
    return np.array([2 * np.pi * np.sum(w * t * legendre.legval(t, [0] * n + [1]) * np.sqrt((2 * n + 1) / (4 * np.pi)))
                     for n in range(n_max + 1)])


HARMONIC_NAMES = [r"$\rho$", r"$\rho n_z$", r"$\rho n_x$", r"$\rho n_y$", r"$\rho(3n_z^2 - 1)$", r"$\rho(n_x^2 - n_y^2)$",
                  r"$\rho n_x n_y$", r"$\rho n_x n_z$", r"$\rho n_y n_z$"]


def harmonic_images(albedo, normals, mask):
    """The nine harmonic images rho, rho n_z, rho n_x, rho n_y, rho (3 n_z^2 - 1), rho (n_x^2 - n_y^2), rho n_x n_y, rho n_x n_z,
    rho n_y n_z of an object (0 outside the mask), as an array (9, n, n); the first four are the orders n <= 1."""
    rho = np.asarray(albedo, dtype=float) * mask
    nx, ny, nz = (np.asarray(normals, dtype=float)[..., i] * mask for i in range(3))
    return np.stack([rho, rho * nz, rho * nx, rho * ny, rho * (3 * nz ** 2 - 1), rho * (nx ** 2 - ny ** 2),
                     rho * nx * ny, rho * nx * nz, rho * ny * nz])


def fit_harmonics(image, basis, mask):
    """Least-squares fit of one image on the given harmonic images (k, n, n) over the mask pixels. Returns the fitted image
    (n, n; 0 outside the mask), the k coefficients and the explained energy 1 - |I - I_hat|^2 / |I|^2."""
    X = np.asarray(basis)[:, mask].T
    I = np.asarray(image, dtype=float)[mask]
    coef = np.linalg.lstsq(X, I, rcond=None)[0]
    fitted = np.zeros(np.shape(image))
    fitted[mask] = X @ coef
    total = np.sum(I ** 2)
    explained = 1.0 - np.sum((I - X @ coef) ** 2) / total if total > 0 else float("nan")
    return fitted, coef, float(explained)


# ------------------------------------------------------------------ 5. Fast marching

def _march_update(T1: float, T2: float, c: float) -> float:
    """The two-case update of the notes with the cost c = h f: both neighbours fixed and |T1 - T2| <= c gives the larger root
    of (T - T1)^2 + (T - T2)^2 = c^2; otherwise the front comes from one neighbour, T = c + min(T1, T2)."""
    if np.isinf(T1) and np.isinf(T2):
        return np.inf
    if np.isinf(T1) or np.isinf(T2) or abs(T1 - T2) > c:
        return c + min(T1, T2)
    return 0.5 * (T1 + T2 + np.sqrt(2 * c * c - (T1 - T2) ** 2))


def fast_march(f, seeds, mask, h: float = 1.0, seed_values=0.0):
    """Fast marching for ||grad T|| = f on a grid of spacing h. T is fixed at the seeds (a boolean array; T = seed_values there,
    one number or an array), and the mask pixels are then fixed in increasing order of T, as in a shortest-path algorithm:
    a pixel with fixed horizontal neighbour T1 and vertical neighbour T2 (the smaller fixed value on each axis) and cost
    c = h f gets T = (T1 + T2 + sqrt(2 c^2 - (T1 - T2)^2)) / 2 if |T1 - T2| <= c, else T = c + min(T1, T2); with one fixed
    neighbour only, T = c + T1. Returns T (n, n), infinite where the front never arrives."""
    f, mask, seeds = np.asarray(f, dtype=float), np.asarray(mask, dtype=bool), np.asarray(seeds, dtype=bool)
    n_rows, n_cols = f.shape
    T = np.full(f.shape, np.inf)
    fixed = np.zeros(f.shape, dtype=bool)
    values = np.broadcast_to(np.asarray(seed_values, dtype=float), f.shape)
    heap = []
    for i, j in zip(*np.nonzero(seeds)):
        T[i, j] = values[i, j]
        heapq.heappush(heap, (float(T[i, j]), int(i), int(j)))

    def fixed_value(i, j):
        return T[i, j] if 0 <= i < n_rows and 0 <= j < n_cols and fixed[i, j] else np.inf

    while heap:
        t, i, j = heapq.heappop(heap)
        if fixed[i, j] or t > T[i, j]:
            continue
        fixed[i, j] = True
        for di, dj in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            a, c = i + di, j + dj
            if not (0 <= a < n_rows and 0 <= c < n_cols) or fixed[a, c] or not mask[a, c]:
                continue
            T1 = min(fixed_value(a, c - 1), fixed_value(a, c + 1))
            T2 = min(fixed_value(a - 1, c), fixed_value(a + 1, c))
            new = _march_update(T1, T2, h * f[a, c])
            if new < T[a, c]:
                T[a, c] = new
                heapq.heappush(heap, (float(new), a, c))
    return T


def eikonal_cost(E):
    """The local cost f = sqrt(1 / E^2 - 1) of shape from shading for a light along the viewing direction and albedo 1
    (E clipped to [1e-3, 1] first)."""
    E = np.clip(np.asarray(E, dtype=float), 1e-3, 1.0)
    return np.sqrt(1.0 / E ** 2 - 1.0)


# ------------------------------------------------------------------ Plotting helpers

def show_image(ax, image, title: str = "", cmap: str = "gray", vmin: float = 0.0, vmax: float = 1.0, mask=None, colorbar: bool = False):
    """One image with y increasing upwards (origin='lower'), a fixed colour scale, and the pixels outside the mask blank."""
    img = np.array(image, dtype=float)
    if mask is not None:
        img = np.where(mask, img, np.nan)
    im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower", extent=(-1, 1, -1, 1), interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    if colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return im


def normal_map(normals, mask):
    """Normals as colours, (n_x, n_y, n_z) -> RGB = (n + 1) / 2 (as on the key sphere of the notes), light grey outside the mask."""
    rgb = 0.5 * (np.asarray(normals, dtype=float) + 1.0)
    rgb[~mask] = 0.85
    return np.clip(rgb, 0.0, 1.0)


def show_normal_map(ax, normals, mask, title: str = ""):
    """The normal map of `normal_map` with the axes of show_image."""
    ax.imshow(normal_map(normals, mask), origin="lower", extent=(-1, 1, -1, 1), interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")


def plot_sv_curves(curves: dict, threshold: float = 1e-6, title: str = ""):
    """D_ii / D_11 against i on a log axis for the models in `curves` ({name: ratios}); blue = linear model, orange = clipped sum."""
    colours = {"linear model": BLUE, "clipped sum": ORANGE}
    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    for name, ratio in curves.items():
        ratio = np.asarray(ratio, dtype=float)
        ax.semilogy(np.arange(1, len(ratio) + 1), np.maximum(ratio, 1e-18), "o-", ms=3, lw=1.5, color=colours.get(name, "k"), label=name)
    ax.axhline(threshold, color="0.5", ls="--", lw=1, label=r"$10^{-6}\,D_{11}$")
    ax.set_xlabel("index $i$")
    ax.set_ylabel("$D_{ii} / D_{11}$")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    plt.show()


def midline_profile(z, x, y):
    """The depth along the vertical midline x = 0 (the column nearest to x = 0), from the bottom (y = -1) to the top (y = 1).
    Returns (y_values, z_values)."""
    col = int(np.argmin(np.abs(x[0])))
    return y[:, col], np.asarray(z, dtype=float)[:, col]


def plot_profiles(profiles: dict, x, y, title: str = "", ylabel: str = "depth"):
    """Midline profiles {label: depth map} against y (bottom to top), as the profiles of the notes."""
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    for (label, z), colour in zip(profiles.items(), ("k", ORANGE, BLUE, TEAL)):
        yy, zz = midline_profile(z, x, y)
        ax.plot(yy, zz, color=colour, lw=1.8, label=label)
    ax.set_xlabel("$y$ along the vertical midline (bottom to top)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    plt.show()

In [ ]:
x, y, z, albedo, mask = make_surface()
h = x[0, 1] - x[0, 0]                                            # the grid spacing
normals = normals_from_depth(z, h)
b = normals * (albedo * mask)[..., None]                         # the scaled normals b = a n, zero on the background
s_point = np.array([np.sin(np.radians(40)), 0.0, np.cos(np.radians(40))])    # a light 40 degrees from the viewing direction, from the right
I_linear, I_clipped = render(b, s_point, clip=False), render(b, s_point)

print(f"{mask.sum()} dome pixels of {mask.size}; grid spacing h = {h:.4f}; depth 0 to {z.max():.3f}")
print(f"under the light s = {np.round(s_point, 3)}: {np.sum(mask & (I_linear < 0))} dome pixels face away from it (attached shadow, "
      f"clipped from {I_linear[mask].min():.3f} to 0); the two images differ only there: {np.sum(mask & (I_linear != I_clipped))} pixels")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), constrained_layout=True)
show_image(axes[0], z, "depth $z$", cmap="viridis", vmin=0, vmax=1, colorbar=True)
show_image(axes[1], albedo, "albedo $a$", vmin=0, vmax=1, mask=mask)
show_normal_map(axes[2], normals, mask, "normals $\\vec n$ as colours")
show_image(axes[3], I_clipped, "image under $s$ (clipped sum)", vmin=0, vmax=1)
plt.show()

## Rank three, then not

Stack $N$ images of the $p$ dome pixels as the columns of $J$ ($p \times N$). Under the linear model $J = BS$, with the scaled normals as the rows of $B$ ($p \times 3$) and the lights as the columns of $S$ ($3 \times N$). The singular value decomposition $J = UDV^T$ ($D_{11} \ge D_{22} \ge \dots$) gives the best fit of rank three, and the leftover $\sum_{i \ge 4} D_{ii}^2$ tests the model: it is zero exactly when $J = BS$. The cumulative energy $\sum_{i \le k} D_{ii}^2 / \sum_i D_{ii}^2$ is the fraction held by the first $k$ components. The cell uses $N = 40$ lights for the curves and $N = 300$ clipped renderings for the cumulative energies.

In [ ]:
lights_40 = random_lights(40, seed=1)
renderer = (lambda bb, ll: render_cast_shadows(z, bb, ll, x, y)) if OPTIONAL_CAST_SHADOWS else render
J_linear = image_matrix(render(b, lights_40, clip=False), mask)
J_clipped = image_matrix(renderer(b, lights_40), mask)
D_linear, ratio_linear, cum_linear = svd_curve(J_linear)
D_clipped, ratio_clipped, cum_clipped = svd_curve(J_clipped)

print(f"J is {J_linear.shape[0]} pixels x {J_linear.shape[1]} images")
print("D_ii / D_11 for i = 1, ..., 6,  linear model:", np.array2string(ratio_linear[:6], formatter=fmt4))
print("                             clipped sum: ", np.array2string(ratio_clipped[:6], formatter=fmt4))
print(f"singular values above 1e-6 D_11: linear model {np.sum(ratio_linear > 1e-6)}, clipped sum {np.sum(ratio_clipped > 1e-6)}")
print(f"leftover energy sum_(i>=4) D_ii^2 / sum_i D_ii^2: linear model {np.sum(D_linear[3:] ** 2) / np.sum(D_linear ** 2):.3g}, "
      f"clipped sum {np.sum(D_clipped[3:] ** 2) / np.sum(D_clipped ** 2):.4g}")

lights_300 = random_lights(300, seed=2)                              # the whole hemisphere s_3 > 0
D_300, ratio_300, cum_300 = svd_curve(image_matrix(renderer(b, lights_300), mask))
print(f"cumulative energy of the clipped renderings under 300 lights from the whole hemisphere: "
      f"k = 3: {cum_300[2]:.4f}, k = 4: {cum_300[3]:.4f}, k = 9: {cum_300[8]:.4f}; "
      f"99% of the energy needs {components_for(cum_300, 0.99)} components")

plot_sv_curves({"linear model": ratio_linear, "clipped sum": ratio_clipped}, title="singular values of $J$ for $N = 40$ lights, relative to $D_{11}$")

### Question 10.1 (3 points)

- Show the two singular-value curves on one log plot and report, for each model, the number of singular values above $10^{-6} D_{11}$. **(1 point)**
- Report the cumulative energy of the clipped renderings at 3, 4 and 9 components under the 300 lights, next to the lecture's 87.5 % and 99.2 %. **(1 point)**
- Draw 300 lights within $60^\circ$ of the viewing direction (`random_lights(300, min_s3=np.cos(np.radians(60)), seed=3)`) and report the number of components that hold 99 % of the energy, compared with the whole-hemisphere number. **(1 point)**

In [ ]:
# Your code for Question 10.1, e.g.
# lights_front = random_lights(300, min_s3=np.cos(np.radians(60)), seed=3)
# D_front, ratio_front, cum_front = svd_curve(image_matrix(render(b, lights_front), mask))
# print(components_for(cum_front, 0.99), components_for(cum_300, 0.99))

**Your answers for Question 10.1:**

## Photometric stereo, depth from normals and the bas-relief ambiguity

**Known lights.** With $f \ge 3$ lights that are not coplanar, a pixel's intensities $I_j = \vec b \cdot \vec s_j$ are $f$ linear equations in the three unknowns of $\vec b$, solved by least squares. `photometric_stereo` uses at each pixel only the lights under which it is lit ($I_j > 0$) and marks pixels with fewer than three such lights invalid.

**Depth from normals.** The normals of a surface have $n_1 / n_3 = -z_x$, $n_2 / n_3 = -z_y$ and satisfy the integrability condition $\partial_x (n_2 / n_3) = \partial_y (n_1 / n_3)$, whose residual `integrability_residual` computes. `depth_from_slopes` recovers $z$ by least squares on the slope equations, with $p = -n_1 / n_3$ and $q = -n_2 / n_3$:

$$\min_z \sum_{x, y} \big( z(x{+}1, y) - z(x, y) - h\,p \big)^2 + \big( z(x, y{+}1) - z(x, y) - h\,q \big)^2 ,$$

with the slopes multiplied by the grid spacing $h$ of the pixels (the code takes the mean slope of the two pixels of each pair).

**The ambiguity.** With unknown lights, $\vec b \mapsto A^T \vec b$ with $\vec s \mapsto A^{-1} \vec s$ leaves every image unchanged for any invertible $A$. The generalised bas-relief (GBR) transformation $z' = \lambda' z + \mu x + \nu y$ ($\mu, \nu$ constants here, not the light index) acts on the scaled normals as $\vec b \mapsto M \vec b$ and moves the lights to $M^{-T} \vec s = G \vec s / \lambda$; `gbr` applies it with $\tau = 1$: $M$ has rows $(\lambda', 0, -\mu)$, $(0, \lambda', -\nu)$, $(0, 0, 1)$ and $G$ rows $(1, 0, 0)$, $(0, 1, 0)$, $(\mu, \nu, \lambda')$. The cell renders four known lights within $40^\circ$ of the viewing direction, runs photometric stereo and the integration, and computes the residual of the true normals, after a random $A$ and after the GBR with $\lambda' = 0.6$, $\mu = 0.2$, $\nu = -0.15$.

In [ ]:
lights_4 = np.array([[0.0, 0.0, 1.0], [0.5, 0.0, 0.866], [0.0, 0.5, 0.866], [-0.4, -0.3, 0.866]])
lights_4 /= np.linalg.norm(lights_4, axis=1, keepdims=True)
images_4 = render(b, lights_4)
albedo_ps, normals_ps, valid, b_ps = photometric_stereo(images_4, lights_4, mask)
angle = angular_error(normals_ps, normals, valid)
print(f"photometric stereo with {len(lights_4)} lights: {valid.sum()} valid pixels, {np.sum(mask & ~valid)} dome pixels with fewer than three "
      f"unshadowed lights left out; RMS angular error of the normals {rms(angle, where=valid):.3g} degrees, "
      f"max |albedo error| {np.max(np.abs(albedo_ps - albedo)[valid]):.3g}")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), constrained_layout=True)
show_image(axes[0], albedo, "true albedo", vmin=0, vmax=1, mask=mask)
show_image(axes[1], albedo_ps, "recovered albedo $|\\vec b|$", vmin=0, vmax=1, mask=valid)
show_normal_map(axes[2], normals, mask, "true normals")
show_normal_map(axes[3], normals_ps, valid, "recovered normals $\\vec b / |\\vec b|$")
plt.show()

# depth from the recovered normals (valid pixels only), against the true depth, both with their means removed
p_ps = np.divide(-normals_ps[..., 0], normals_ps[..., 2], out=np.zeros(z.shape), where=valid)
q_ps = np.divide(-normals_ps[..., 1], normals_ps[..., 2], out=np.zeros(z.shape), where=valid)
z_rec = depth_from_slopes(p_ps, q_ps, valid, h)
z_true_centred = np.where(valid, z - z[valid].mean(), np.nan)
print(f"depth from the slopes: RMS difference from the true depth over the valid pixels {rms(z_rec, z_true_centred, where=valid):.4g} "
      f"(the depth range there is {np.ptp(z[valid]):.3f})")
vmax_z = float(np.nanmax(np.abs(z_true_centred)))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
show_image(axes[0], z_true_centred, "true depth (mean removed)", cmap="viridis", vmin=-vmax_z, vmax=vmax_z, mask=valid, colorbar=True)
show_image(axes[1], z_rec, "depth from the slopes (mean removed)", cmap="viridis", vmin=-vmax_z, vmax=vmax_z, mask=valid, colorbar=True)
show_image(axes[2], z_rec - z_true_centred, "difference", cmap=RESIDUAL_CMAP, vmin=-0.02, vmax=0.02, mask=valid, colorbar=True)
plt.show()

# the integrability residual: true normals, after a random A, after the GBR (inside the dome, away from its steep rim)
rng = np.random.default_rng(5)
A = np.eye(3) + 0.5 * rng.normal(size=(3, 3))
b_A = b @ A                                                   # (A^T b) at every pixel
lam_p, mu, nu = 0.6, 0.2, -0.15
z_gbr, b_gbr, lights_gbr = gbr(z, b, lights_4, x, y, lam_p, mu, nu)
interior = ndimage.binary_erosion(mask) & (normals[..., 2] > 0.5)
residuals = {"true normals": integrability_residual(normals, interior, h),
             "after the random A": integrability_residual(unit(b_A), interior, h),
             "after the GBR": integrability_residual(unit(b_gbr), interior, h)}
for name, r in residuals.items():
    print(f"integrability residual, {name:19s}: RMS {rms(r[interior]):.4g}, largest |residual| {np.nanmax(np.abs(r)):.4g}  ({interior.sum()} pixels)")
scale = float(np.nanpercentile(np.abs(residuals["after the random A"][interior]), 99))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
for ax, (name, r) in zip(axes, residuals.items()):
    im = show_image(ax, r, name.replace(" A", " $A$"), cmap=RESIDUAL_CMAP, vmin=-scale, vmax=scale, mask=interior)
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label=r"$\partial_x(n_2/n_3) - \partial_y(n_1/n_3)$")
plt.show()
print("A =")
print(A)
print(f"GBR with lambda' = {lam_p}, mu = {mu}, nu = {nu}: the lights G s / lambda =")
print(lights_gbr)

### Question 10.2 (5 points)

- Show the recovered albedo and normal maps beside the true ones and report the root-mean-square (RMS) angular error of the normals over the valid pixels. **(1 point)**
- Show the depth recovered from the slopes and the true depth, both with their means removed, and report the RMS difference. **(1 point)**
- Show the integrability residual for the true normals, after the random $A$ and after the GBR, three panels on one colour scale, and report the RMS of each. **(1 point)**
- Re-render the four images from $(M \vec b,\ G \vec s / \lambda)$ and report the largest absolute difference from the originals. Which identity makes it zero up to rounding? **(1 point)**
- Plot the depth profiles along the vertical midline before and after the GBR, with your $\lambda', \mu, \nu$ printed, and report the two peak heights. **(1 point)**

In [ ]:
# Your code for Question 10.2, e.g.
# images_gbr = render(b_gbr, lights_gbr)
# print(np.max(np.abs(images_gbr - images_4)))
# plot_profiles({"$z$": z, f"$\\lambda' z + \\mu x + \\nu y$ with ({lam_p}, {mu}, {nu})": z_gbr}, x, y)

**Your answers for Question 10.2:**

## The Lambertian kernel and the nine harmonic images

The Lambertian kernel $k(\theta) = \max(\cos\theta, 0) = \sum_n k_n Y_{n0}(\theta)$ has the coefficients $k_n = \int k\,Y_{n0}$ and the energy $\int_{S^2} k^2 = 2\pi/3 = \sum_n k_n^2$ (`kernel_coefficients`, by numerical integration). The image of an object with albedo $\rho$ and normals $\vec n$ under any lighting is approximately a combination of the nine harmonic images

$$\rho,\quad \rho n_z,\ \rho n_x,\ \rho n_y,\quad \rho(3n_z^2 - 1),\ \rho(n_x^2 - n_y^2),\ \rho n_x n_y,\ \rho n_x n_z,\ \rho n_y n_z$$

(the first four are the orders $n \le 1$). The cell computes $k_n$ for $n \le 8$ with the cumulative fractions $\sum_{n \le N} k_n^2 / (2\pi/3)$, builds the nine images of the dome, and fits its clipped rendering under the light $\vec s$ of the first cell by least squares on all nine; the explained energy of a fit is $1 - \|I - \hat I\|^2 / \|I\|^2$.

In [ ]:
k_n = kernel_coefficients(n_max=8)
cumulative_fraction = np.cumsum(k_n ** 2) / KERNEL_ENERGY
print("k_n for n = 0, ..., 8:", np.array2string(np.round(k_n, 4) + 0.0, formatter={"float_kind": lambda v: f"{v:.4f}"}))
print("cumulative fraction of the kernel energy through order N = 0, ..., 8:", np.array2string(cumulative_fraction, formatter=fmt4))
print(f"orders <= 1 hold {100 * cumulative_fraction[1]:.4g} % of the energy, orders <= 2 hold {100 * cumulative_fraction[2]:.4g} %")

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.bar(np.arange(9), k_n, color=TEAL, width=0.6)
ax.axhline(0, color="k", lw=0.8)
ax.annotate("", xy=(-0.4, 1.15), xytext=(2.4, 1.15), arrowprops=dict(arrowstyle="|-|", color="crimson", lw=1.5))
ax.text(1.0, 1.2, "orders 0-2", color="crimson", ha="center", fontsize=9)
ax.set_xlabel("order $n$")
ax.set_ylabel("$k_n$")
ax.set_title("coefficients of the Lambertian kernel $\\max(\\cos\\theta, 0)$", fontsize=10)
ax.set_ylim(-0.25, 1.35)
plt.show()

H_images = harmonic_images(albedo, normals, mask)
scale_h = float(np.max(np.abs(H_images)))
fig, axes = plt.subplots(3, 3, figsize=(10, 9.5), constrained_layout=True)
for ax, img, name in zip(axes.ravel(), H_images, HARMONIC_NAMES):
    im = show_image(ax, img, name, cmap=RESIDUAL_CMAP, vmin=-scale_h, vmax=scale_h, mask=mask)
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
plt.show()

I_point = render(b, s_point)                                       # the clipped rendering under the point light of the first cell
fit_9, coef_9, explained_9 = fit_harmonics(I_point, H_images, mask)
print(f"fit of the clipped rendering under s = {np.round(s_point, 3)} with 9 harmonic images: explained energy {explained_9:.5f}; "
      f"coefficients {np.array2string(coef_9, formatter=fmt4)}")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
show_image(axes[0], I_point, "image under $s$ (clipped sum)", vmin=0, vmax=1)
show_image(axes[1], fit_9, "fit with 9 harmonic images", vmin=0, vmax=1, mask=mask)
show_image(axes[2], I_point - fit_9, "residual of the 9-image fit", cmap=RESIDUAL_CMAP, vmin=-0.05, vmax=0.05, mask=mask, colorbar=True)
plt.show()

### Question 10.3 (3 points)

- Show your $k_n$ for $n = 0, \dots, 8$ as a bar chart and report the cumulative fractions of the kernel energy at the orders $N \le 1$ and $N \le 2$. **(1 point)**
- Show the nine harmonic images of the dome in a $3 \times 3$ panel with a common colour scale. **(1 point)**
- Fit the clipped rendering with 4 and with 9 harmonic images: report the explained energy of each and show the residual image of the 9-image fit. **(1 point)**

In [ ]:
# Your code for Question 10.3, e.g.
# fit_4, coef_4, explained_4 = fit_harmonics(I_point, H_images[:4], mask)
# print(explained_4, explained_9)

**Your answers for Question 10.3:**

## Fast marching: the distance transform and shape from shading

Under a light along the viewing direction, $\vec l = (0, 0, 1)$, with albedo 1, the image $E$ of a depth map obeys $E^2 (z_x^2 + z_y^2 + 1) = 1$, the eikonal equation $\|\nabla z\| = f$ with $f = \sqrt{1/E^2 - 1}$. The distance transform of a region is the case $f = 1$ with $z = 0$ on the boundary. Fast marching solves $\|\nabla T\| = f$ on the grid as a shortest-path algorithm does, fixing the nodes in increasing order of $T$; a node with fixed neighbours $T_1$ (horizontal) and $T_2$ (vertical) gets

$$T = \tfrac12 \Big( T_1 + T_2 + \sqrt{2 f^2 - (T_1 - T_2)^2} \Big) \ \text{ if } |T_1 - T_2| \le f, \qquad T = f + \min\{T_1, T_2\} \ \text{ otherwise}$$

(cost $h f$ on a grid of spacing $h$). For shape from shading the boundary condition is the depth at the local maxima of intensity, where $f = 0$: from the brightest pixel the recovered depth is $z_{\max} - T$, and the hollow twin $z_{\min} + T$ takes the opposite sign there; recall the bump-and-dip identity $\vec n(-z) \cdot \vec s\,' = \vec n(z) \cdot \vec s$ for $\vec s\,' = (-s_1, -s_2, s_3)$. The cell compares the distance transform of a disc united with a rectangle with `scipy.ndimage.distance_transform_edt`, then recovers the smooth dome (`make_surface(bumps=False)`) from its image: every local maximum of intensity would need its own boundary depth, so the bumps are left out (you may switch them on). The rim's slopes exceed what the $64 \times 64$ grid resolves, so the RMS is taken over the pixels with $E > 0.2$.

In [ ]:
# (a) the distance transform of a binary shape: fast marching with f = 1 from the background, against scipy
rows, cols = np.mgrid[0:N_GRID, 0:N_GRID]
shape = ((rows - 36) ** 2 + (cols - 26) ** 2 <= 20 ** 2) | ((rows >= 20) & (rows < 34) & (cols >= 30) & (cols < 60))   # disc radius 20, 30 x 14 rectangle
T_shape = fast_march(np.ones(shape.shape), seeds=~shape, mask=shape, h=1.0)
dt_scipy = ndimage.distance_transform_edt(shape)
difference = np.where(shape, T_shape - dt_scipy, 0.0)
worst = np.unravel_index(np.argmax(np.abs(difference)), difference.shape)
print(f"distance transform of the shape ({shape.sum()} pixels): largest distance {T_shape[shape].max():.3f} (fast marching) vs "
      f"{dt_scipy.max():.3f} (scipy); largest |difference| {np.abs(difference).max():.4g} pixels at (row, column) = {tuple(int(v) for v in worst)}, "
      f"where the fast-marching distance is {T_shape[worst]:.3f} and scipy's {dt_scipy[worst]:.3f}; RMS difference {rms(difference, where=shape):.4g}")
vmax_T = float(dt_scipy.max())
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
for ax, img, title in zip(axes[:2], (T_shape, dt_scipy), ("fast marching, $f = 1$", "scipy distance_transform_edt")):
    im = ax.imshow(np.where(shape, img, np.nan), cmap="viridis", vmin=0, vmax=vmax_T, origin="lower", interpolation="nearest")
    ax.set_title(title, fontsize=10); ax.set_xlabel("column"); ax.set_ylabel("row")
fig.colorbar(im, ax=axes[:2], fraction=0.02, pad=0.02, label="distance (pixels)")
im = axes[2].imshow(np.where(shape, difference, np.nan), cmap=RESIDUAL_CMAP, vmin=-1, vmax=1, origin="lower", interpolation="nearest")
axes[2].set_title("fast marching $-$ scipy", fontsize=10); axes[2].set_xlabel("column"); axes[2].set_ylabel("row")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04, label="pixels")
plt.show()

# (b) shape from shading of the smooth dome under l = (0, 0, 1), albedo 1
x_s, y_s, z_smooth, _, mask_smooth = make_surface(bumps=False)
E_smooth = render(normals_from_depth(z_smooth, h), np.array([0.0, 0.0, 1.0]))      # albedo 1: E = n_3
f_smooth = eikonal_cost(E_smooth)
seed = np.zeros(mask_smooth.shape, dtype=bool)
seed[np.unravel_index(np.argmax(np.where(mask_smooth, E_smooth, -1.0)), E_smooth.shape)] = True       # the brightest dome pixel
T_dome = fast_march(f_smooth, seeds=seed, mask=mask_smooth, h=h)
z_max = float(z_smooth[seed][0])                                                     # the boundary depth at the brightest pixel
z_dome = np.where(np.isfinite(T_dome), z_max - T_dome, np.nan)
region = mask_smooth & (E_smooth > 0.2) & np.isfinite(T_dome)
z_dome_centred = z_dome - np.nanmean(z_dome[region])
z_smooth_centred = z_smooth - z_smooth[region].mean()
print(f"shape from shading of the smooth dome: brightest pixel at (row, column) = {tuple(int(v) for v in np.argwhere(seed)[0])}, E = {E_smooth[seed][0]:.4f}, "
      f"z_max = {z_max:.4f}; RMS difference from the true depth over the {region.sum()} pixels with E > 0.2 (means removed) {rms(z_dome_centred, z_smooth_centred, where=region):.4g}")
plot_profiles({"true depth $z$": z_smooth_centred, "recovered $z_{\\max} - T$": np.where(region, z_dome_centred, np.nan)}, x_s, y_s,
              title="the smooth dome along the vertical midline (means removed; pixels with $E > 0.2$)")

### Question 10.4 (3 points)

- Show the two distance transforms and their difference, and report the largest absolute difference and where it occurs. **(1 point)**
- Show the recovered dome against the true depth along the midline (means removed) and report the RMS difference. **(1 point)**
- Show the hollow twin's profile next to the dome's and report the largest difference between the images that the twin and the recovered dome $z_{\max} - T$ render under $\vec l = (0, 0, 1)$. **(1 point)**

In [ ]:
# Your code for Question 10.4, e.g.
# z_twin = np.where(np.isfinite(T_dome), -z_max + T_dome, np.nan)                    # z_min + T with z_min = -z_max
# E_dome = render(normals_from_depth(np.nan_to_num(z_dome), h), np.array([0.0, 0.0, 1.0]))
# E_twin = render(normals_from_depth(np.nan_to_num(z_twin), h), np.array([0.0, 0.0, 1.0]))
# print(np.max(np.abs(E_dome - E_twin)[region]))
# plot_profiles({"dome $z_{\\max} - T$": z_dome, "hollow twin $z_{\\min} + T$": z_twin}, x_s, y_s)

**Your answers for Question 10.4:**